# Experimento 2A: PubMedBERT + typed entity markers

**Objetivo:** aislar el efecto de una unica tecnica -- marcadores de entidad tipados
(`[HEAD-tipo] ... [/HEAD-tipo]`, `[TAIL-tipo] ... [/TAIL-tipo]`) insertados en el
texto -- manteniendo TODO lo demas identico al baseline de PubMedBERT (1A/1G,
seed 42): mismos hiperparametros, mismos datos (mismo neg_ratio=3 ya aplicado en
`eng_train.txt`), mismo protocolo de evaluacion ciega (`baseline/score.py`).

**Sobre el historial "Exp1-Exp9" de otros notebooks (`5-experimento-classweights.ipynb`,
`9-experimento-pubmedbert-large.ipynb`, `1C-experimento-xlm-roberta.ipynb`):
NO se usan ni se citan aqui.** Se comprobo que:
- El numero "Exp1 PubMedBERT = 0.7754" que citan ya se demostro fabricado/no
  verificable (commit `446a038`, corregido en 1A/1B a 0.7777).
- No existen notebooks `2-`, `3-` ni `4-` en el repo que generaran Exp2/Exp3/Exp4.
- No hay ningun checkpoint ni `results_*.json` local para Exp2-Exp5 ni Exp9.

Este notebook se construye y verifica desde cero. La comparacion se hace contra
los UNICOS numeros de PubMedBERT que si estan verificados localmente (checkpoint
+ json reales): `outputs/1A-pubmedbert/seed42/results_seed_summary.json`
(argmax ciego = 0.3162) y `outputs/multiseed/multiseed_finegrid_resultados.csv`
(F1 calibrado fino = 0.4316 @ threshold=0.997).

## 1. Setup

In [1]:
# Ejecucion en servidor local (zape), entorno conda "tfg". Mismo patron que 1G.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

!python ../baseline/patch_opennre.py


HF cache: /home/lucia.esperon/hf_cache/hub


Traceback (most recent call last):
  File "/home/lucia.esperon/TFG/proyecto/mi-tfg-bionner/notebooks/../baseline/patch_opennre.py", line 153, in <module>
    patch()
    ~~~~~^^
  File "/home/lucia.esperon/TFG/proyecto/mi-tfg-bionner/notebooks/../baseline/patch_opennre.py", line 26, in patch
    import opennre.framework.data_loader as dl
ModuleNotFoundError: No module named 'opennre'


In [2]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric
add_macro_f1_metric()
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")


/home/lucia.esperon/miniconda3/envs/tfg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu124 | CUDA: True
GPU: NVIDIA GeForce RTX 2080 Ti (11.5 GB)


## 2. Configuracion -- identica a 1A/1G salvo la tecnica

In [3]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_typed_markers"
TECHNIQUE       = "typed entity markers ([HEAD-tipo]/[TAIL-tipo]) -- unico cambio vs 1A/1G"

MAX_LENGTH     = 256   # identico al baseline -- NO se sube, para aislar el efecto de los marcadores
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0   # igual que 1G (evita picos de loss)

DATA_DIR    = Path("../data/english")
TRAIN_DATA  = DATA_DIR / "eng_train.txt"   # ya tiene neg_ratio=3 aplicado, sin tocar
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/2A-pubmedbert-typed/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)


Clases: 15 | modelo: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Salida: ../outputs/2A-pubmedbert-typed/seed42


## 3. Marcadores tipados -- construccion y verificacion

Inserta `[HEAD-tipo] ... [/HEAD-tipo]` y `[TAIL-tipo] ... [/TAIL-tipo]` alrededor
de cada entidad, recalculando sus posiciones en el texto nuevo. Los marcadores
quedan FUERA del span de la entidad: OpenNRE sigue insertando sus propios
`[unused0]-[unused3]` justo alrededor del texto original de la entidad (eso no
cambia), y los marcadores tipados dan contexto textual justo fuera de esos
unused tokens.

**Antes de aplicarlo a los datos reales, se verifica con casos sinteticos** que
el texto de la entidad se recupera exactamente en las posiciones nuevas --
incluyendo el caso limite de entidades adyacentes sin hueco entre ellas.

In [4]:
def build_typed_text(text, h_start, h_end, t_start, t_end, h_type, t_type):
    """Devuelve (texto_nuevo, [nuevo_h_start, nuevo_h_end], [nuevo_t_start, nuevo_t_end]).
    En empates de posicion (entidades adyacentes) los cierres van antes que las
    aperturas, para no anidar un marcador dentro de otro.
    """
    HEAD_OPEN, HEAD_CLOSE = f"[HEAD-{h_type}] ", f" [/HEAD-{h_type}]"
    TAIL_OPEN, TAIL_CLOSE = f"[TAIL-{t_type}] ", f" [/TAIL-{t_type}]"

    events = [
        (h_start, 1, HEAD_OPEN, "h_start"),
        (h_end,   0, HEAD_CLOSE, "h_end"),
        (t_start, 1, TAIL_OPEN, "t_start"),
        (t_end,   0, TAIL_CLOSE, "t_end"),
    ]
    events.sort(key=lambda x: (x[0], x[1]))

    parts, prev, new_pos = [], 0, {}
    for pos, _, marker, tag in events:
        parts.append(text[prev:pos])
        if tag in ("h_start", "t_start"):
            parts.append(marker)
            new_pos[tag] = sum(len(p) for p in parts)
        else:
            new_pos[tag] = sum(len(p) for p in parts)
            parts.append(marker)
        prev = pos
    parts.append(text[prev:])
    new_text = "".join(parts)
    return new_text, [new_pos["h_start"], new_pos["h_end"]], [new_pos["t_start"], new_pos["t_end"]]


# --- verificacion con casos sinteticos, incluyendo el limite (adyacentes) ---
_tests = [
    ("idiopathic generalized epilepsies (IGE) suffered aggravation", (0, 34), (36, 39), "DISO", "DISO"),
    ("IGE was aggravated by idiopathic generalized epilepsies", (23, 57), (0, 3), "DISO", "DISO"),
    ("headtail rest", (0, 4), (4, 8), "CHEM", "DISO"),
    ("tailhead rest", (4, 8), (0, 4), "DISO", "CHEM"),
]
for text, h, t, ht, tt in _tests:
    orig_h, orig_t = text[h[0]:h[1]], text[t[0]:t[1]]
    new_text, nh, nt = build_typed_text(text, h[0], h[1], t[0], t[1], ht, tt)
    assert new_text[nh[0]:nh[1]] == orig_h, f"FALLO head: {new_text!r}"
    assert new_text[nt[0]:nt[1]] == orig_t, f"FALLO tail: {new_text!r}"
print(f"{len(_tests)}/{len(_tests)} casos sinteticos OK: la entidad se recupera exacta en las posiciones nuevas.")


4/4 casos sinteticos OK: la entidad se recupera exacta en las posiciones nuevas.


## 4. Aplicar los marcadores a train / dev / blind

In [5]:
def load_instances(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


def apply_typed_markers(instances):
    out = []
    n_checked_ok = 0
    for inst in instances:
        hs, he = inst["h"]["pos"]
        ts, te = inst["t"]["pos"]
        new_text, new_h, new_t = build_typed_text(
            inst["text"], hs, he, ts, te, inst["head_type"], inst["tail_type"])
        new_inst = dict(inst)
        new_inst["text"] = new_text
        new_inst["h"] = dict(inst["h"]); new_inst["h"]["pos"] = new_h
        new_inst["t"] = dict(inst["t"]); new_inst["t"]["pos"] = new_t
        # verificacion en dato real: el span nuevo debe recuperar el nombre original
        if new_text[new_h[0]:new_h[1]] == inst["text"][hs:he] and \
           new_text[new_t[0]:new_t[1]] == inst["text"][ts:te]:
            n_checked_ok += 1
        out.append(new_inst)
    print(f"  verificacion en datos reales: {n_checked_ok}/{len(instances)} spans recuperados exactos")
    return out


train_instances = load_instances(TRAIN_DATA)
dev_instances = load_instances(DEV_DATA)
print(f"train: {len(train_instances)} | dev: {len(dev_instances)}")

print("Aplicando marcadores tipados a train...")
train_typed = apply_typed_markers(train_instances)
print("Aplicando marcadores tipados a dev...")
dev_typed = apply_typed_markers(dev_instances)

TRAIN_TYPED = OUT_DIR / "eng_train_typed.txt"
DEV_TYPED = OUT_DIR / "eng_dev_typed.txt"
with open(TRAIN_TYPED, "w", encoding="utf-8") as f:
    for inst in train_typed:
        f.write(json.dumps(inst, ensure_ascii=False) + "\n")
with open(DEV_TYPED, "w", encoding="utf-8") as f:
    for inst in dev_typed:
        f.write(json.dumps(inst, ensure_ascii=False) + "\n")
print("Guardado:", TRAIN_TYPED, DEV_TYPED)

print("\nEjemplo:")
print("  original:", train_instances[0]["text"][:120])
print("  typed:   ", train_typed[0]["text"][:140])


train: 12739 | dev: 2967
Aplicando marcadores tipados a train...
  verificacion en datos reales: 11035/12739 spans recuperados exactos
Aplicando marcadores tipados a dev...
  verificacion en datos reales: 1450/2967 spans recuperados exactos


Guardado: ../outputs/2A-pubmedbert-typed/seed42/eng_train_typed.txt ../outputs/2A-pubmedbert-typed/seed42/eng_dev_typed.txt

Ejemplo:
  original: First was topiramate (TPM) (n=12), followed by valproates (VPA) (n=8), carbamazepine (CBZ) (n=5), lamotrigine (LTG) (n=1
  typed:    First was topiramate (TPM) (n=12), followed by valproates (VPA) (n=8), carbamazepine (CBZ) (n=5), lamotrigine (LTG) (n=1) and levetiracetam 


In [6]:
# Cuanto alarga los textos (relevante por si trunca contra MAX_LENGTH=256) --
# no se sube MAX_LENGTH para mantener el experimento aislado a un solo cambio,
# esto es solo diagnostico.
from transformers import BertTokenizer
_tok = BertTokenizer.from_pretrained(MODEL_NAME)
_sample = random.Random(0).sample(train_typed, min(500, len(train_typed)))
_lens_before = [len(_tok.tokenize(i2["text"])) for i2 in random.Random(0).sample(train_instances, min(500, len(train_instances)))]
_lens_after = [len(_tok.tokenize(i2["text"])) for i2 in _sample]
print(f"tokens (muestra de 500): antes media={np.mean(_lens_before):.1f} p95={np.percentile(_lens_before,95):.0f} | "
      f"despues media={np.mean(_lens_after):.1f} p95={np.percentile(_lens_after,95):.0f}")
print(f">{MAX_LENGTH} tokens tras marcadores: {sum(1 for l in _lens_after if l > MAX_LENGTH)}/{len(_lens_after)}")


2026-07-29 09:42:18,506 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"


2026-07-29 09:42:18,509 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-07-29 09:42:18,541 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/tokenizer_config.json "HTTP/1.1 200 OK"


2026-07-29 09:42:18,707 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


2026-07-29 09:42:18,856 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


tokens (muestra de 500): antes media=184.3 p95=401 | despues media=208.1 p95=424
>256 tokens tras marcadores: 142/500


## 5. Entrenamiento -- misma funcion que 1G (`train_with_history`)

In [7]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    """Identica a la de 1G: valida y guarda el mejor checkpoint por macro_f1
    en cada epoch, devuelve el historial."""
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history


In [8]:
set_seed(SEED)

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_TYPED), val_path=str(DEV_TYPED), test_path=str(DEV_TYPED),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev typed)={macro_f1_curado:.4f}")


2026-07-29 09:42:19,357 - root - INFO - Loading BERT pre-trained checkpoint.


2026-07-29 09:42:19,512 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-07-29 09:42:19,543 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/e1354b7a3a09615f6aba48dfad4b7a613eef7062/config.json "HTTP/1.1 200 OK"


2026-07-29 09:42:19,761 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


2026-07-29 09:42:19,945 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


...

 99%|█████████▉| 184/186 [00:11<00:00, 15.84it/s, acc=0.807]

100%|██████████| 186/186 [00:11<00:00, 16.88it/s, acc=0.807]

100%|██████████| 186/186 [00:11<00:00, 16.26it/s, acc=0.807]


2026-07-29 10:20:06,813 - root - INFO - Evaluation result: {'acc': 0.8068756319514662, 'micro_p': 0.9071618037135278, 'micro_r': 0.8068756319514662, 'micro_f1': 0.8540849090260435}.


Epoch 14: loss=0.0059 val_micro_f1=0.8541 val_macro_f1=0.7912
Mejor macro_f1 en val: 0.8035

Entreno: 37.7 min | mejor epoch=9 macro_f1_curado(dev typed)=0.8035


## 6. Inferencia en blind (typed) + evaluacion oficial (argmax)

In [9]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

print("Aplicando marcadores tipados al blind...")
blind_typed = []
for inst in blind_raw:
    hs, he = inst["h"]["pos"]; ts, te = inst["t"]["pos"]
    new_text, new_h, new_t = build_typed_text(
        inst["text"], hs, he, ts, te, inst["head_type"], inst["tail_type"])
    ni = dict(inst)
    ni["text"] = new_text
    ni["h"] = dict(inst["h"]); ni["h"]["pos"] = new_h
    ni["t"] = dict(inst["t"]); ni["t"]["pos"] = new_t
    blind_typed.append(ni)
print(f"listo: {len(blind_typed)} candidatos con marcadores tipados")


candidatos blind: 282364 | gold real: 2891
Aplicando marcadores tipados al blind...


listo: 282364 candidatos con marcadores tipados


In [10]:
# Recargar el mejor checkpoint y predecir en blind (typed), en lotes -- igual
# patron que 1G/0-validacion-blind-oficial.
model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_typed), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_typed), 64):
        batch = blind_typed[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")


Inferencia blind: 22.6 min


Macro F1 ciego (argmax puro): 0.3262


## 7. Calibracion de threshold (grid fino) -- comparacion justa contra el baseline

In [11]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def f1_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)["macro_f1"]

# mismo grid fino que baseline/fine_resweep.py
FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, f1_at(all_probs, th)) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

# --- baseline verificado (1G, seed 42) -- fuente exacta en cada numero ---
BASELINE_ARGMAX = 0.31624749637347316      # outputs/1A-pubmedbert/seed42/results_seed_summary.json -> macro_f1_ciego_argmax
BASELINE_CALIBRADO = 0.4315666494418448    # outputs/multiseed/multiseed_finegrid_resultados.csv -> pubmedbert,42
BASELINE_TH = 0.997

print(f"\n{'':<28}{'argmax':>10}{'calibrado':>12}{'threshold':>12}")
print(f"{'PubMedBERT baseline (1G)':<28}{BASELINE_ARGMAX:>10.4f}{BASELINE_CALIBRADO:>12.4f}{BASELINE_TH:>12.3f}")
print(f"{'+ typed markers':<28}{macro_f1_ciego_argmax:>10.4f}{macro_f1_ciego_calibrado:>12.4f}{best_threshold:>12.3f}")
print(f"{'delta':<28}{macro_f1_ciego_argmax-BASELINE_ARGMAX:>+10.4f}{macro_f1_ciego_calibrado-BASELINE_CALIBRADO:>+12.4f}")


Mejor threshold (fino): 0.9990 -> Macro F1 ciego calibrado = 0.4134

                                argmax   calibrado   threshold
PubMedBERT baseline (1G)        0.3162      0.4316       0.997
+ typed markers                 0.3262      0.4134       0.999
delta                          +0.0100     -0.0182


## 8. Guardar resultados

In [12]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": 3, "seed": SEED,
    },
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "baseline_comparison": {
        "source": "outputs/1A-pubmedbert/seed42/results_seed_summary.json + outputs/multiseed/multiseed_finegrid_resultados.csv",
        "baseline_macro_f1_ciego_argmax": BASELINE_ARGMAX,
        "baseline_macro_f1_ciego_calibrado": BASELINE_CALIBRADO,
        "delta_argmax": macro_f1_ciego_argmax - BASELINE_ARGMAX,
        "delta_calibrado": macro_f1_ciego_calibrado - BASELINE_CALIBRADO,
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
print(json.dumps(results, indent=2, ensure_ascii=False))


Guardado: ../outputs/2A-pubmedbert-typed/seed42/results_seed_summary.json
{
  "exp": "pubmedbert_typed_markers",
  "model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
  "technique": "typed entity markers ([HEAD-tipo]/[TAIL-tipo]) -- unico cambio vs 1A/1G",
  "seed": 42,
  "hyperparameters": {
    "max_length": 256,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 15,
    "warmup_steps": 300,
    "neg_ratio": 3,
    "seed": 42
  },
  "macro_f1_curado": 0.803536679567353,
  "macro_f1_ciego_argmax": 0.32619839962112557,
  "best_threshold_fino": 0.999,
  "macro_f1_ciego_calibrado": 0.4134121241306147,
  "en_borde_del_grid_fino": false,
  "train_minutes": 37.7,
  "blind_minutes": 22.6,
  "baseline_comparison": {
    "source": "outputs/1A-pubmedbert/seed42/results_seed_summary.json + outputs/multiseed/multiseed_finegrid_resultados.csv",
    "baseline_macro_f1_ciego_argmax": 0.31624749637347316,
    "baseline_macro_f1_ciego_calibrado": 0.4315666494418448,


## 9. Conclusion

*(rellenar tras ejecutar con el resultado real -- no se pone ningun numero aqui
de antemano, a diferencia del historial Exp1-Exp9 de otros notebooks)*